# Transaction Foundation Model on Ray — Part 1: Setup

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~5 min

### Anyscale Technical Demo — Ray Data + Ray Train + Ray Serve

---

## What we're building

Transaction foundation models are the latest generation of transformer models — like LLMs, but instead of language, they are trained on financial transactions and similar timeseries record data. This lets them recognize patterns, like fraud, that traditional ML techniques can't detect. Stripe, Visa, Mastercard, Nubank, and Revolut all run models like this in production.

This series of notebooks shows you how to build your own transaction foundation model, with performance and scalability that surpass [NVIDIA's blueprint](https://github.com/NVIDIA-AI-Blueprints/transaction-foundation-model), using an identical approach but with Ray achieving a scale that the NVIDIA blueprint cant reach. Here, we reuse NVIDIA's tokenizer, train their exact architecture with their recipe, and score fraud with their same evaluation method. So its an apples-to-apples comparison. Ray is the only difference. Along the way we maximize our GPUs, solve bottlenecks through independently scalable CPUs, and save money using a fault-tollerant cluster of AWS Spot instances. 

This is what a production workload looks like. In comparison, NVIDIA's repo runs every stage on a single node, and it never actually trains the foundation model — its tains a small 30 steps demonstration, and then downloads the real weights. This template runs the actual training: ~16,000 steps, or 8 epochs over all 19.5 million training transactions, about two hours on eight GPUs. Dont worry, with the exact same set of notebooks, you can scale down just as easily as you scale up - see our 'mini.yaml' config file for a small cpu-only configuration of this exact same code.

## 

## How we're building it

This series builds a complete fraud detection system, showcasing the alternative approaches a team can take with a transaction foundation model.

In this Part 1 notebook, we get oriented and set up the cluster and dependencies. Part 2 downloads the dataset and prepares it for training and testing. The dataset is IBM's TabFormer benchmark, a 24 million transaction credit card purchase database. Each transaction comes with 13 raw fields (amount, merchant, location, time, and so on). Those raw fields are what a typical fraud team uses in their traditional XGBoost based classifiers today.

Parts 3 & 4 pretrain the **foundation model** itself: a transformer trained on 19.5 million training transactions to predict each card's next transaction. This unlabeled training allows us to build a model that learns the data's underlying patterns without having to manually label transactions as fraud (or not). This is the same way an LLM pre-trains on large volumes of plain natural language text.

After the pre-training, the rest of the series builds **five fraud detectors**, each using the foundation model in more powerful ways than the last.

1. **raw** (Part 6) — XGBoost on the 13 raw fields alone. No foundation model involved, and no feature engineering either: a production fraud team would add engineered features (spending velocity, per-card aggregates) and land higher, but we keep NVIDIA's exact protocol so the comparison with the later approaches stays apples-to-apples. Read raw as a floor, not as what a mature pipeline achieves.
2. **embedding** (Parts 5 & 6) — XGBoost on the foundation model's **embedding**: run a transaction through the model and it returns 512 numbers encoding what the model understands about that transaction. This is the most straightforward way for XGBoost to use a foundation model. Part 5 performs the embeddings, Part 6 classifies fraud with them. 
3. **fusion (raw + embedding)** (Part 6) — XGBoost uses the traditional "raw" fields and also the embedding. The idea here is that the two signals identify fraud in different ways, so combined they are more powerful.
4. **fine-tuned** (Part 7) — Finally, removing XGBoost entirely, basing the model purely on a Transformer model. We put a classification head on the foundation model and train its weights on the fraud labels directly, the way an LLM gets fine-tuned for a task. The most modern approach here. If this approach beats our fusion model, it allows an organization to completely remove the fragile, ad-hoc, manually built xgboost based pipeline.
5. **ensemble (raw + fine-tuned)** (Part 7) — the fine-tuned model's score combined with the raw fields, for teams that keep their existing pipeline. It scores highest of all, but it retains exactly the xgboost pipeline the fine-tuned approach exists to remove.

One foundation model, five detectors — using it not at all, as a feature factory, as extra features, as the classifier itself, and as an ensemble with what you already have. Every performance number in this series is one of these five scored on the same held-out transactions.

## Model performance

Every model outputs a fraud score for each transaction. We evaluate how well the model performs by evaluating two metrics: **Average Precision (AP)** and **AUC-ROC**. 
- AP measures how often the model is correct when it alerts that a transaction is fraudulent — a perfect model scores 1.0, a random one scores 0.001 here.
- AUC-ROC measures how reliably fraud scores higher than normal transactions.

AP is most critical here. AP tells you how correct it is with high-confidence fruad predictions. These are the predictions that require an intervention like disabling a customer's credit card. It's critical to be correct. 

AUC-ROC is less useful, partly because all of the models have similar scores. It is just not a distingishing evaluation metric. A model with a high AP but lower AUC-ROC will still correctly identify high-confidence fruadulent transactions, but will not do as well on low-confident ones. 

For both of these metrics we evluate our model on the exact same 100K-transaction test set NVIDIA uses.

**AP (Average precision):**

| model | NVIDIA AP | our AP | notes |
|---|---|---|---|
| raw | 0.1238 | 0.1238 | Raw XGBoost model, no feature engineering. The baseline model |
| embedding | 0.0123 | 0.04–0.06 | Foundation Model embeddings w/ XGBoost. Anyscale scored 3–5× NVIDIA |
| fusion (raw + embedding) | 0.1755 | 0.136 - 0.284 | Foundation Model + Raw model. NVIDIA gave a single self-selected score, we report the full range|
| fine-tuned | — | 0.1263 | Our fine tuned foundation model. No XGboost, no feature engineering pipeline needed. NVIDIA does not provide this model|
| ensemble (raw + fine-tuned) | — | 0.1988 | Top score creates an ensemble of the raw xgboost and fine tuned foundation models |

**AUC-ROC:**

| model | NVIDIA AUC-ROC | our AUC-ROC | notes |
|---|---|---|---|
| raw | 0.989 | 0.9885 |  |
| embedding | 0.878 | 0.96 |  |
| fusion (raw + embedding) | 0.993 | 0.9916 |  |
| fine-tuned | — | 0.9513 |  |
| ensemble (raw + fine-tuned) | — | 0.9923 |  |

Three results matter here. Our foundation model's embedding scores 3–5× higher than NVIDIA's. Neither embeddings based model scores well enough to replace the raw xgboost model. But the fusion model outperforms. This was the takeaway headline in NVIDIA's blueprint.

Meanwhile, we take things further. Our fine-tuned model — pure transformer, no XGBoost, no raw feature pipeline — matches the raw baseline on its own. This means an organization could retire the hand-built pipeline entirely with a transformer based model with no impact on fraud detection performance. If they chose to keep the traditional feature engineering pipeline in an ensemble with the the fine tuned model, they can achieve best of class results of 0.1988 AP.

So to recap here, the approach we take going forward is identical in approach to NVIDIA up to the fusion model, yet we outperform them in our evaluation. Why? We're not doing any interesting data science tricks or creating some research paper level algorithm. **It's simply because Ray let's us scale.** 

## Scalability

The only difference between NVIDIA's foundation model and ours is training time. Their published model trained for about 3,000 steps — roughly one pass over the data — and their repo doesn't even do that much: the training notebook runs a 30-step demonstration and downloads the real weights. We ran the same architecture, on the same data, with the same recipe, for 8 full passes. That took two hours on eight GPUs, and it's where the 3–5× embedding win comes from. The fine-tuned models are two more training runs at about 30 minutes each. On a single machine, each of those runs is a day of blocked hardware; on a cluster they're routine, so you actually run them. That is what "Ray lets us scale" means in practice.

The cluster earns its cost by putting each stage on the hardware it needs. Data preparation runs on cheap CPU workers with Ray Data. Training runs on GPUs with Ray Train. Embedding extraction uses both at once: CPU workers tokenize transactions and stream them to GPU workers running the model, and the two pools scale independently, so a GPU never sits idle waiting on data work.

Distributing someone else's pipeline can quietly change its results, so we verified ours didn't. Every distributed stage produces output identical to NVIDIA's single-machine implementation: the same 19.5 million rows in the same order, a bit-for-bit identical training corpus, byte-equal labels and features. The checks are in `scripts/verify_*.py`, and the raw baseline matching to the fourth decimal place is the end-to-end proof.

The cost model follows from the elasticity. Worker nodes come up when a stage requests them — 2 to 3 minutes in our runs — and scale back to zero when the stage finishes, so across a complete run the GPUs exist for about three hours total. The blueprint's single always-on A100 node pays for its GPU the whole time, whether it's training, preparing data, or idle.

## Architecture

The whole series runs in one Anyscale workspace attached to one elastic cluster. The cluster has three kinds of nodes: a head node that runs the notebooks and deliberately schedules no work, a CPU worker group for the data stages, and a GPU worker group for training and inference. Stages hand artifacts to each other through shared cluster storage, which is why you can run Part 4 today and Part 5 tomorrow — or skip a stage entirely when its output already exists.

```
                     ┌─────────────────────── Anyscale ────────────────────────┐
 IBM TabFormer ────► │ [Ray Data]   temporal split + corpus (CPU workers)      │
 (24M txns)          │ [Ray Train]  next-token pretraining, 8 GPUs (DDP)       │
                     │ [Ray Data]   embeddings: CPU tokenize → GPU actors      │
                     │ [XGBoost]    fraud: raw vs embedding vs fusion (GPU)    │
                     │ [Ray Train]  fine-tune: fraud head on the model, 8 GPUs │
                     │ [Ray Serve]  online embedding + fraud score             │
                     └─────────────────────────────────────────────────────────┘
```

Every stage is the same code at every size — you change `ScalingConfig`, not your program. One `SCALE` variable at the top of each notebook picks a config file under `configs/`; diff `mini.yaml` against `full.yaml` and you are looking at the complete difference between the CPU-only sanity run and the full eight-GPU build. There is no other switch.

## The series

We build the foundation model end-to-end, one stage per notebook:

| # | Notebook | Runs on |
|---|----------|---------|
| **1** | **Setup** *(this notebook)* | — |
| 2 | Load & explore the data | Ray Data, CPU workers |
| 3 | Tokenize — build the pretrain corpus | Ray Data, CPU workers |
| 4 | Pretrain — next-token prediction | Ray Train, 8 GPUs |
| 5 | Batch embedding extraction | Ray Data, CPU → GPU actors |
| 6 | Downstream fraud — raw vs embedding vs fusion | XGBoost on GPU |
| 7 | Fine-tune the foundation model | Ray Train, 8 GPUs |
| 8 | Online serving | Ray Serve |
| 9 | Run the whole pipeline as a job | Anyscale Jobs |
| 10 | Scaling up — the bottlenecks you'll hit | All of the above |

## What a full run needs

`mini` proves the plumbing on CPU in minutes. The results above come from `SCALE = "full"`, which needs:

- the IBM TabFormer dataset — a one-time ~2.3 GB download that Part 2 handles,
- GPU workers — 8×A10G for the ~2h pretraining, for embedding extraction, and for about an hour of Part 7's fine-tuning (two variants, ~30 min each); the downstream XGBoost also needs a GPU (Part 6 explains why the result depends on it),
- a CPU worker group in the cluster config, so the CPU-heavy data stages don't pull up GPU nodes just for their CPUs (Part 10 shows that failure mode).

Nothing else changes between the two scales. Same notebooks, same code, different config file.

---

## Step 1: Install dependencies

The cell below installs the template's dependencies and registers them on every cluster node, so the same imports resolve on workers as well as the head node. Two pins matter. `xgboost` is 3.2.0 because the downstream fraud result is sensitive to an early-stopping behavior that changed in 3.3. RAPIDS (`cudf`) is NVIDIA's GPU tokenizer, kept as the reference implementation — the notebooks run a CPU implementation we verified byte-identical to it, which is why `mini` needs no GPU at all.

> In production you'd install from the generated `python_depset.lock`. Here we install from `requirements.txt` for portability.

In [1]:
!pip install -q -r requirements.txt

Successfully registered `ray, torch` and 13 other packages to be installed on all cluster nodes.
View and update dependencies here: https://console.anyscale.com/cld_g54aiirwj1s8t9ktgzikqur41k/prj_f1j47h9srml4cyg962id75ms2e/workspaces/expwrk_78mtwtucrd61tjxf851krarzwr?workspace-tab=dependencies


## Step 2: Attach to the cluster

In an Anyscale Workspace, Ray is already running — `ray.init()` **attaches** to the workspace's cluster rather than starting one. `working_dir` ships this template's `src/` package to every worker, so the notebook and remote tasks/actors all import the same code.

In [2]:
import sys, os
import logging


# Make the template's `src/` package importable from the notebook.
DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import ray

# In an Anyscale Workspace, Ray is already running — this attaches to it.
ray.init(ignore_reinit_error=True, runtime_env={"working_dir": DEMO_ROOT},   logging_level=logging.ERROR)

from src.utils import print_cluster_resources
print_cluster_resources()

Ray cluster resources:
  anyscale/cpu_only:true 1.0
  anyscale/node-group:head 1.0
  anyscale/provider:aws 1.0
  anyscale/region:us-west-2 1.0
  memory               34359738368.0
  object_store_memory  9599240601.0

Cluster nodes: 1
  10.0.158.37          alive=True  anyscale/cpu_only:true=1.0, anyscale/region:us-west-2=1.0, anyscale/provider:aws=1.0, memory=34359738368.0, object_store_memory=9599240601.0, anyscale/node-group:head=1.0


/home/ray/anaconda3/lib/python3.12/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


You should see the head node and its resources. On a fresh or idle cluster the head node intentionally schedules no work — GPU and CPU worker nodes autoscale up when later stages request them, and scale back down when idle. You don't manage that; Ray does.

That's the whole setup: dependencies registered, cluster attached. Every notebook from here starts the same way and picks up where the previous one left off, using artifacts written to shared storage.

---

## Next

**Part 2 — Load & explore the data**: download the IBM TabFormer benchmark, rebuild NVIDIA's temporal train/val/test split from the raw CSV with Ray Data, and see in the data why a 0.1% fraud rate dictates how the whole series measures success.